# Classification NBA Model

## Configuration

## Imports

In [47]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training
)
from sklearn.model_selection import KFold, cross_validate, train_test_split
from xgboost import XGBClassifier


In [48]:
from nba_ou.modeling.modeling import (
    split_latest_dates_holdout,
    make_walk_forward_last_n_seasons_splits,
    validate_time_splits,
    make_test_anchored_walk_forward_splits,
    assert_valid_time_splits,
)

In [49]:
import time
time.sleep(60*25)

## Load Data

In [50]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260311.csv"

path = data_path + name

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)
df_stats['GAME_DATE'] = pd.to_datetime(df_stats['GAME_DATE']).dt.strftime('%Y-%m-%d')

In [51]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=50, max_na_per_row=6, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'])

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 11196 rows
Basic cleaning complete: 8587 rows remaining

Starting advanced column cleaning with 3097 columns

Advanced column cleaning complete: 3097 → 2084 columns (1013 removed)


Dropping NA rows for SEASON_YEAR 2017...
   Removed 0 rows with NaN values from 2017 season

Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 0 (0.0%)
  Critical columns requiring data: 4
  Columns zero-filled: 98
  Infer pairs applied: 20/136
  Remaining NaN cells: 944087

Dropping rows with more than 6 NaN values...
Removed 3922 rows exceeding NaN threshold
CLEANING COMPLETE
Final shape: (4665, 2084)


In [52]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_to_train[df_to_train[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_to_train.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year
1893,total_consensus_pct_under,72,1.54,2024.0
1894,spread_consensus_pct_home,67,1.44,2024.0
1749,total_consensus_pct_over_TREND_SLOPE_LAST_5_HO...,15,0.32,2023.0
1755,spread_consensus_pct_home_TREND_SLOPE_LAST_5_H...,15,0.32,2023.0
1901,ml_consensus_opener_price_home,14,0.30,2025.0
1751,total_consensus_pct_under_TREND_SLOPE_LAST_5_H...,14,0.30,2023.0
1900,ml_consensus_opener_price_away,14,0.30,2025.0
1430,total_betmgm_price_under_LAST_HOME_AWAY_5_MATC...,12,0.26,2021.0
1427,total_betmgm_price_over_LAST_HOME_AWAY_5_MATCH...,12,0.26,2021.0
1457,spread_betmgm_price_LAST_HOME_AWAY_5_MATCHES_D...,12,0.26,2021.0


In [53]:
BET365_LINE_COL =  "TOTAL_LINE_bet365"
# BET365_LINE_COL =  "total_bet365_line_over"

# Ensure scoring line and target exist (avoid NaN-driven undefined betting accuracy).
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()

In [54]:
df_to_train['GAME_DATE'] = pd.to_datetime(df_to_train['GAME_DATE'])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

In [55]:
#count games per season
games_per_season = df_to_train.groupby('SEASON_YEAR').size()
print(games_per_season)

SEASON_YEAR
2021    1171
2022    1174
2023     179
2024    1248
2025     893
dtype: int64


## Train / Test

In [56]:
from sklearn.model_selection import KFold, cross_validate, cross_val_predict, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, make_scorer, root_mean_squared_error
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.base import clone

In [57]:
df_dev, df_test_final = split_latest_dates_holdout(
    df=df_to_train,
    date_col="GAME_DATE",
    test_size=0.05,
)

print(f"Development set size: {len(df_dev)}")
print(f"Final test set size: {len(df_test_final)}")
print("Final test date range:",
      df_test_final["GAME_DATE"].min(), "->", df_test_final["GAME_DATE"].max())

Development set size: 4430
Final test set size: 235
Final test date range: 2026-02-02 00:00:00 -> 2026-03-10 00:00:00


In [58]:
TARGET_COL = "TOTAL_POINTS"

EXCLUDE_COLS = [
    "TOTAL_POINTS",
    "SEASON_YEAR",
    "GAME_DATE",
]

X_dev = df_dev.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(df_dev[TARGET_COL], errors="coerce")

X_test_final = df_test_final.drop(columns=EXCLUDE_COLS, errors="ignore")
y_test_final = pd.to_numeric(df_test_final[TARGET_COL], errors="coerce")

print(f"X_dev shape: {X_dev.shape}")
print(f"X_test_final shape: {X_test_final.shape}")

X_dev shape: (4430, 2081)
X_test_final shape: (235, 2081)


In [59]:
from nba_ou.modeling.scorers import OverUnderScorerTotalPoints, over_under_betting_accuracy_total_points, evaluate_total_points_thresholds
    
ou_scorer = OverUnderScorerTotalPoints(BET365_LINE_COL)

scoring = {
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2",
    "OU_Betting_Accuracy": ou_scorer,
}

def print_metrics(cv_results):
    for sc in scoring.keys():
        train_key = f"train_{sc}"
        test_key = f"test_{sc}"

        train_val = cv_results[train_key].mean()
        test_val = cv_results[test_key].mean()

        if sc in {"MSE", "RMSE", "MAE"}:
            train_val = -train_val
            test_val = -test_val

        if sc == "OU_Betting_Accuracy":
            print(f"Train {sc}: {train_val:.2%}")
            print(f"Validation {sc}: {test_val:.2%}")
        else:
            print(f"Train {sc}: {train_val:.5f}")
            print(f"Validation {sc}: {test_val:.5f}")
        print()

In [60]:
splits, fold_info = make_test_anchored_walk_forward_splits(
    df=df_dev,
    date_col="GAME_DATE",
    season_col="SEASON_YEAR",
    test_games=30,
    step_games_between_tests=100,
    train_games=3500,
    min_train_games=2000,
    max_folds=12,
    verbose=1,
)


Created 12 test-anchored walk-forward folds
 fold  train_n_games  test_n_games train_start_date train_end_date test_start_date test_end_date  test_season
    1           2814            40       2021-10-29     2024-12-11      2024-12-12    2024-12-19         2024
    2           2959            31       2021-10-29     2025-01-03      2025-01-04    2025-01-07         2024
    3           3097            39       2021-10-29     2025-01-22      2025-01-23    2025-01-27         2024
    4           3242            34       2021-10-29     2025-02-10      2025-02-11    2025-02-20         2024
    5           3378            30       2021-10-29     2025-03-05      2025-03-06    2025-03-09         2024
    6           3500            30       2021-10-30     2025-03-22      2025-03-23    2025-03-26         2024
    7           3500            35       2021-11-20     2025-04-09      2025-04-10    2025-04-13         2024
    8           3500            37       2021-12-09     2025-10-31      2025

In [61]:
season_bl = DummyRegressor(strategy="mean")

cv_results = cross_validate(
    season_bl,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("DummyRegressor baseline")
print_metrics(cv_results)

DummyRegressor baseline
Train MSE: 380.70878
Validation MSE: 356.22633

Train RMSE: 19.51132
Validation RMSE: 18.76610

Train MAE: 15.56968
Validation MAE: 15.21894

Train R2: 0.00000
Validation R2: -0.05558

Train OU_Betting_Accuracy: 49.56%
Validation OU_Betting_Accuracy: 52.39%



In [62]:
lr = LinearRegression()

cv_results = cross_validate(
    lr,
    X_dev.fillna(0),   # LR cannot handle NaNs
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

print("Linear Regression")
print_metrics(cv_results)

Linear Regression
Train MSE: 164.25461
Validation MSE: 30270.73463

Train RMSE: 12.80934
Validation RMSE: 87.72115

Train MAE: 9.95397
Validation MAE: 40.06349

Train R2: 0.56821
Validation R2: -150.12603

Train OU_Betting_Accuracy: 73.40%
Validation OU_Betting_Accuracy: 53.74%



In [63]:
xgb_reg = XGBRegressor(
    max_depth=3,
    learning_rate=0.05,
    n_estimators=350,
    subsample=0.65,
    colsample_bytree=0.68,
    reg_alpha=5.28,
    reg_lambda=1.3,
    min_child_weight=5.08,
    gamma=0.0085,
    n_jobs=-2,          # choose your CPU count
    random_state=16,
)

cv_results = cross_validate(
    xgb_reg,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,          # leave at 1 because XGB already parallelizes internally
)

print("XGBoost")
print_metrics(cv_results)

XGBoost
Train MSE: 125.66543
Validation MSE: 309.55081

Train RMSE: 11.20237
Validation RMSE: 17.48146

Train MAE: 8.89116
Validation MAE: 14.17743

Train R2: 0.66960
Validation R2: 0.07103

Train OU_Betting_Accuracy: 85.43%
Validation OU_Betting_Accuracy: 49.28%



In [64]:
xgb_reg.fit(X_dev, y_dev)

y_pred_test_total = xgb_reg.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_test_final,
    y_pred_test_total,
    betting_line,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")

Final test metrics
MSE: 313.99753
RMSE: 17.71997
MAE: 13.93019
OU_Betting_Accuracy: 53.48%


In [65]:
results_df, y_pred_test_total = evaluate_total_points_thresholds(
    model=xgb_reg,
    X_test=X_test_final,
    y_test_total=y_test_final,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_betting_accuracy": "{:.2%}"}
    )
)

,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy
0,0,235,100.0%,53.48%
1,1,175,74.5%,55.23%
2,2,135,57.4%,58.65%
3,3,95,40.4%,54.26%
4,4,64,27.2%,57.14%
5,5,39,16.6%,55.26%
6,6,25,10.6%,60.00%
7,7,11,4.7%,54.55%
8,8,4,1.7%,50.00%
9,9,2,0.9%,50.00%


# OPTUNA

In [66]:
from nba_ou.modeling.optuna_total_points import (
    tune_xgb_total_points_optuna,
    summarize_optuna_trials,
    fit_best_xgb_total_points,
)

study = tune_xgb_total_points_optuna(
    X=X_dev,
    y=y_dev,
    splits=splits,
    line_col=BET365_LINE_COL,
    n_trials=80,
    timeout=5 * 3600,
    objective_name="reg:squarederror",   # second run: reg:pseudohubererror
    study_name="xgb_total_points_mae",
)

print("Best CV MAE:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")

print("\nSecondary metrics from best trial:")
print("Mean RMSE:", study.best_trial.user_attrs.get("mean_rmse"))
print("Mean R2:", study.best_trial.user_attrs.get("mean_r2"))
print("Mean OU accuracy:", study.best_trial.user_attrs.get("mean_ou_acc"))
print("Mean best_iteration:", study.best_trial.user_attrs.get("mean_best_iteration"))

trials_df = summarize_optuna_trials(study)
display(
    trials_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
        }
    )
)

[I 2026-03-11 18:58:53,819] A new study created in memory with name: xgb_total_points_mae


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-03-11 19:05:32,885] Trial 0 finished with value: 13.473320999626303 and parameters: {'max_depth': 2, 'min_child_weight': 14.839989016734002, 'gamma': 1.6521043697435434, 'subsample': 0.5682407800531226, 'colsample_bytree': 0.6123279759064977, 'learning_rate': 0.015902381379451283, 'reg_alpha': 1.8771791376898666, 'reg_lambda': 0.2766344129879233}. Best is trial 0 with value: 13.473320999626303.
[I 2026-03-11 19:14:50,913] Trial 1 finished with value: 13.407725687198045 and parameters: {'max_depth': 2, 'min_child_weight': 35.38241645406617, 'gamma': 1.6910441407044758, 'subsample': 0.5811969357559948, 'colsample_bytree': 0.7751882300061779, 'learning_rate': 0.013902617405829187, 'reg_alpha': 0.06701717253228541, 'reg_lambda': 0.6196026989845951}. Best is trial 1 with value: 13.407725687198045.
[I 2026-03-11 19:20:38,178] Trial 2 finished with value: 13.26499760677671 and parameters: {'max_depth': 4, 'min_child_weight': 13.12932060704323, 'gamma': 0.6451864311430185, 'subsample':

,trial,value_mae,mean_rmse,mean_r2,mean_ou_acc,mean_best_iteration,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda
0,57,13.1089,16.6643,0.1621,58.65%,110,4,5.484121,1.770088,0.863413,0.861852,0.057114,0.574475,1.718720
1,59,13.1654,16.6766,0.1629,57.44%,97,4,5.689454,2.218022,0.871615,0.878360,0.053732,0.501144,1.157654
2,49,13.1820,16.7221,0.1558,56.45%,101,4,6.801828,1.927764,0.709687,0.898387,0.058040,1.063996,1.550630
3,52,13.2085,16.8554,0.1424,59.02%,88,4,5.803861,2.127613,0.789211,0.899022,0.058179,0.615098,1.639094
4,55,13.2398,16.7446,0.1523,58.37%,67,4,6.919842,2.290749,0.769320,0.898032,0.074142,0.511099,4.888402
5,51,13.2607,16.8624,0.1403,59.55%,76,4,6.819507,2.044784,0.789474,0.892052,0.056813,0.685538,1.697780
6,2,13.2650,16.6725,0.1597,60.77%,63,4,13.129321,0.645186,0.728731,0.504395,0.067415,0.741065,1.879170
7,58,13.2652,16.7985,0.1484,56.29%,82,4,5.221448,2.275678,0.858113,0.860562,0.056550,0.590895,7.257108
8,42,13.2893,16.8985,0.1382,58.65%,98,3,11.153248,1.503759,0.736284,0.751829,0.064972,3.276548,1.580652
9,6,13.2895,16.7583,0.1539,57.22%,143,5,7.825674,1.069222,0.557360,0.727556,0.016146,2.821794,3.774625


In [67]:
best_model = fit_best_xgb_total_points(
    X_dev=X_dev,
    y_dev=y_dev,
    study=study,
    objective_name="reg:squarederror",
)

y_pred_test_total = best_model.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")

Final test metrics
MSE: 313.36130
RMSE: 17.70201
MAE: 13.99785
OU_Betting_Accuracy: 55.22%


In [68]:
import json
import joblib
from pathlib import Path


def save_model_bundle(
    model: XGBRegressor,
    feature_names: list[str],
    out_dir: str | Path,
    name: str = "xgb_total_points_model",
    extra_meta: dict | None = None,
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    model_path = out_dir / f"{name}.joblib"
    meta_path = out_dir / f"{name}.meta.json"

    joblib.dump(model, model_path)

    metadata = {
        "feature_names": list(feature_names),
        "xgboost_version_note": "Saved via joblib XGBRegressor wrapper",
        **(extra_meta or {}),
    }
    meta_path.write_text(json.dumps(metadata, indent=2))

    return model_path, meta_path


def load_model_bundle(model_path: str | Path, meta_path: str | Path):
    model = joblib.load(model_path)
    metadata = json.loads(Path(meta_path).read_text())
    return model, metadata


model_path, meta_path = save_model_bundle(
    model=best_model,
    feature_names=list(X_dev.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/",
    name="recent_xgb_total_points_11_03_26",
    extra_meta={
        "best_params": study.best_trial.params,
        "mean_best_iteration": study.best_trial.user_attrs.get("mean_best_iteration"),
        "cv_mae": study.best_value,
        "cv_rmse": study.best_trial.user_attrs.get("mean_rmse"),
        "cv_ou_acc": study.best_trial.user_attrs.get("mean_ou_acc"),
        "final_test_mae": float(mae),
        "final_test_rmse": float(rmse),
        "final_test_ou_acc": float(ou_acc),
    },
)

print("Saved model :", model_path)
print("Saved metadata:", meta_path)


Saved model : /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/recent_xgb_total_points_11_03_26.joblib
Saved metadata: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/recent_xgb_total_points_11_03_26.meta.json
